In [ ]:
import os
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Optional, Literal, Any

import numpy as np
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

import datasets
import unet
from SplitNet import SplitNet

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def just_darcy(out) -> torch.Tensor:

    # If we assume the output is in order k,pres,phi
    # pres_grad is the gradient of the pressure along the y and x directions as a tuple
    pres_grad = torch.gradient(out[:, 1:2], dim=(-2,-1))

    # get velocity by multiplying the gradient by the conductivity
    y_grad = pres_grad[0] * out[:, 0:1]
    x_grad = pres_grad[1] * out[:, 0:1]

    # compute the divergence by the second derivative of the gradients and adding them together
    yy_grad = torch.gradient(y_grad, spacing=(1,),dim=(-2,))[0]
    xx_grad = torch.gradient(x_grad, spacing=(1,),dim=(-1,))[0]
    final = yy_grad + xx_grad

    # total divergence should be 0
    loss = (final**2)

    return loss.mean()

In [ ]:
ModelType = Literal["splitnet_attn", "splitnet", "unet", "attn_unet"]
DatasetMode = Literal["border", "fixed", "random"]
TrainingMode = Literal["physics_limited", "baseline_full"]

@dataclass
class ExperimentConfig:
    model_type: ModelType = "splitnet_attn"
    dataset_mode: DatasetMode = "border"

    # physics_limited:
    #   Uses Limited datasets: (input_sample, sparse_target, mask)
    #   MSE trains only on mask points.
    #   Darcy loss is applied over the full model output.
    #
    # baseline_full:
    #   Uses Full datasets: (input_sample, full_target)
    #   MSE trains over the whole output.
    #   Darcy loss must be 0.
    training_mode: TrainingMode = "physics_limited"

    mse_weight: float = 1.0
    darcy_weight: float = 1.0

    epochs: int = 250
    batch_size: int = 8
    lr: float = 1e-3

    channels: str = "KP"
    train_sims_path: str = "../train_sims.npy"
    val_sims_path: str = "../val_sims.npy"
    sim_max_exclusive: Optional[int] = 500

    save_prefix: Optional[str] = None
    save_best: bool = True
    save_final: bool = True

    # For physics_limited, this chooses how masked MSE is computed.
    # "old_zeroed" matches your old script:
    #     crit(out * mask, label * mask)
    # This divides by the whole image size, including zeros outside the mask.
    # "true_masked" divides only by the number of masked pixels.
    mask_loss_style: str = "old_zeroed"

    # If None, run_experiment chooses a sensible default:
    #     physics_limited -> val_supervised_mse
    #     baseline_full    -> val_total_mse
    best_metric: Optional[str] = None

    # Optional dataset overrides, e.g.
    # dataset_kwargs={"points_per_side": 5, "radius": 3, "steps": (0, 200)}
    dataset_kwargs: Optional[Dict[str, Any]] = None


# Model
# ----------------


def make_model(model_type: ModelType = "splitnet_attn", channels: str = "KP") -> nn.Module:
    """
    Builds a model matching the selected channel setup.

    For channels='KP', all models output 2 channels: K and P.
    SplitNet already outputs K and P, so it is only compatible with channels='KP'.
    """
    model_type = model_type.lower()

    if channels == "all":
        num_channels = 3
    elif channels == "KP":
        num_channels = 2
    elif channels in ["K", "P", "phi"]:
        num_channels = 1
    else:
        raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")

    if model_type == "splitnet_attn":
        if channels != "KP":
            raise ValueError("SplitNet is designed for channels='KP'.")
        return SplitNet(attn=True).to(DEVICE)

    if model_type == "splitnet":
        if channels != "KP":
            raise ValueError("SplitNet is designed for channels='KP'.")
        return SplitNet(attn=False).to(DEVICE)

    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)

    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")

In [ ]:
# Dataset
# --------------

def _dataset_class(dataset_mode, training_mode):
    """
    Chooses the dataset class for the experiment.

    training_mode='physics_limited':
        Uses Limited datasets. These return (sample, target, mask).
        The target is sparse/limited, and the mask tells us where supervised MSE is allowed.

    training_mode='baseline_full':
        Uses Full datasets.
        These return (sample, full_target).
        Supervised MSE is computed over the full output.
    """
    if training_mode == "physics_limited":
        if dataset_mode == "border":
            return datasets.BorderThinDatasetLimited
        if dataset_mode == "fixed":
            return datasets.FixedThinDatasetLimited
        if dataset_mode == "random":
            return datasets.RandomDenseDatasetLimited

    if training_mode == "baseline_full":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetFull
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetFull
        if dataset_mode == "random":
            raise ValueError("There is no RandomDenseDatasetFull")
    raise ValueError("Invalid dataset")


def make_loaders(config) -> Tuple[DataLoader, DataLoader]:
    train_sims = np.load(config.train_sims_path)
    train_sims = train_sims[train_sims < config.sim_max]
    val_sims = np.load(config.val_sims_path)
    val_sims = val_sims[val_sims < config.sim_max]

    dataset_class = _dataset_class(config.dataset_mode, config.training_mode)

    kwargs = dict(config.dataset_kwargs or {})
    kwargs["channels"] = config.channels

    train_data = dataset_class(train_sims, **kwargs)
    val_data = dataset_class(val_sims, **kwargs)

    train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=config.batch_size, shuffle=False)

    return train_loader, val_loader


# Loss / metric
# ------------


def align_channels(label: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    """
    Keeps label channels compatible with model output channels.
    Example: if a dataset gives 3 channels but the model outputs KP only, keep K and P.
    """
    if label.shape[1] == out.shape[1]:
        return label
    if label.shape[1] > out.shape[1]:
        return label[:, : out.shape[1]]
    raise ValueError(f"Label has {label.shape[1]} channels but output has {out.shape[1]} channels.")


def expand_mask_for_channels(mask: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    if mask.dim() == 3:
        mask = mask.unsqueeze(1)
    return mask.expand(-1, out.shape[1], -1, -1)


def mse_on_region(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    True masked MSE. This divides by number of masked pixels, not by whole image size.
    """
    mask = mask.float()
    diff2 = ((pred - target) ** 2) * mask
    denom = mask.sum()
    if denom.item() == 0:
        return torch.tensor(0.0, device=pred.device)
    return diff2.sum() / denom


def supervised_loss(
    out: torch.Tensor,
    label: torch.Tensor,
    mask: Optional[torch.Tensor],
    training_mode: TrainingMode,
    crit: nn.Module,
    mask_loss_style: str = "old_zeroed",
) -> torch.Tensor:
    """
    baseline_full:
        MSE over the whole output.

    physics_limited:
        MSE only over mask points.

        mask_loss_style='old_zeroed' matches your old script exactly:
            crit(out * mask, label * mask)

        mask_loss_style='true_masked' computes a mathematically true masked MSE:
            sum((out-label)^2 over mask) / number_of_mask_pixels
    """
    label = align_channels(label, out)

    if training_mode == "baseline_full":
        return crit(out, label)

    if training_mode == "physics_limited":
        if mask is None:
            raise ValueError("physics_limited mode requires a mask.")
        mask_c = expand_mask_for_channels(mask.bool(), out).float()

        if mask_loss_style == "old_zeroed":
            return crit(out * mask_c, label * mask_c)

        if mask_loss_style == "true_masked":
            return mse_on_region(out, label, mask_c)

        raise ValueError("mask_loss_style must be 'old_zeroed' or 'true_masked'.")

    raise ValueError(f"Unknown training_mode: {training_mode}")

def unpack_batch(batch, training_mode: TrainingMode):
    if training_mode == "physics_limited":
        feat, label, mask = batch
        return feat, label, mask
    if training_mode == "baseline_full":
        feat, label = batch
        return feat, label, None
    raise ValueError(f"Unknown training_mode: {training_mode}")

In [ ]:

# Evaluation
# -----------

def evaluate_loader(model: nn.Module, loader: DataLoader, config: ExperimentConfig, crit: nn.Module) -> Dict[str, float]:
    model.eval()

    total_mse = 0.0
    supervised_mse = 0.0
    mask_mse = 0.0
    nonmask_mse = 0.0
    darcy = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)
            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            out = model(feat)
            label = align_channels(label, out)

            total_mse += crit(out, label).item()
            supervised_mse += supervised_loss(
                out, label, mask, config.training_mode, crit, config.mask_loss_style
            ).item()
            darcy += just_darcy(out).item()

            if mask is not None:
                mask_c = expand_mask_for_channels(mask, out)
                nonmask_c = ~mask_c
                mask_mse += mse_on_region(out, label, mask_c).item()
                nonmask_mse += mse_on_region(out, label, nonmask_c).item()
            else:
                mask_mse += float("nan")
                nonmask_mse += float("nan")

            n_batches += 1

    return {
        "total_mse": total_mse / n_batches,
        "supervised_mse": supervised_mse / n_batches,
        "mask_mse": mask_mse / n_batches,
        "nonmask_mse": nonmask_mse / n_batches,
        "darcy": darcy / n_batches,
    }

In [ ]:
def run_experiment(**kwargs):
    """
    Main experiment entry point.

    Physics run:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="physics_limited",
            mse_weight=1.0,
            darcy_weight=0.1,
            epochs=250,
            save_prefix="minimum_info/fixed_splitattn_darcy_0p1"
        )

    No-physics baseline:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="baseline_full",
            mse_weight=1.0,
            darcy_weight=0.0,
            epochs=250,
            save_prefix="minimum_info/fixed_splitattn_baseline"
        )
    """
    config = ExperimentConfig(**kwargs)

    if config.training_mode == "baseline_full" and config.darcy_weight != 0:
        raise ValueError(
            "baseline_full is the no-physics baseline. Set darcy_weight=0.0, "
            "or use training_mode='physics_limited'."
        )

    if config.training_mode == "physics_limited" and config.darcy_weight == 0:
        print(
            "Warning: physics_limited with darcy_weight=0.0 uses limited/masked MSE but no physics. "
            "For the no-physics full-MSE baseline, use training_mode='baseline_full'."
        )

    if config.save_prefix is not None:
        save_dir = os.path.dirname(config.save_prefix)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

    model = make_model(config.model_type, config.channels)
    optimizer = Adam(model.parameters(), lr=config.lr)
    crit = nn.MSELoss()

    train_loader, val_loader = make_loaders(config)

    history = {
        "train_loss_used": [],
        "train_total_mse": [],
        "train_supervised_mse": [],
        "train_mask_mse": [],
        "train_nonmask_mse": [],
        "train_darcy": [],
        "val_total_mse": [],
        "val_supervised_mse": [],
        "val_mask_mse": [],
        "val_nonmask_mse": [],
        "val_darcy": [],
        "config": asdict(config),
    }

    best_val_score = float("inf")
    best_epoch = 0

    for epoch in tqdm(range(1, config.epochs + 1)):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)
            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            optimizer.zero_grad()

            out = model(feat)
            label = align_channels(label, out)

            mse = supervised_loss(out, label, mask, config.training_mode, crit, config.mask_loss_style)
            physics = darcy_loss_from_output(out)

            loss = config.mse_weight * mse + config.darcy_weight * physics
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        history["train_loss_used"].append(epoch_loss / n_batches)

        train_metrics = evaluate_loader(model, train_loader, config, crit)
        val_metrics = evaluate_loader(model, val_loader, config, crit)

        history["train_total_mse"].append(train_metrics["total_mse"])
        history["train_supervised_mse"].append(train_metrics["supervised_mse"])
        history["train_mask_mse"].append(train_metrics["mask_mse"])
        history["train_nonmask_mse"].append(train_metrics["nonmask_mse"])
        history["train_darcy"].append(train_metrics["darcy"])

        history["val_total_mse"].append(val_metrics["total_mse"])
        history["val_supervised_mse"].append(val_metrics["supervised_mse"])
        history["val_mask_mse"].append(val_metrics["mask_mse"])
        history["val_nonmask_mse"].append(val_metrics["nonmask_mse"])
        history["val_darcy"].append(val_metrics["darcy"])

        # Model selection metric:
        # In physics_limited, total_mse is misleading because the label is sparse/zeroed.
        # supervised_mse is the validation version of the actual training MSE.
        if config.best_metric is not None:
            val_score = val_metrics[config.best_metric]
        elif config.training_mode == "physics_limited":
            val_score = val_metrics["supervised_mse"]
        else:
            val_score = val_metrics["total_mse"]

        if val_score < best_val_score:
            best_val_score = val_score
            best_epoch = epoch
            if config.save_prefix and config.save_best:
                torch.save(model.state_dict(), f"{config.save_prefix}_best_state.pt")

    metric_name = config.best_metric or ("val_supervised_mse" if config.training_mode == "physics_limited" else "val_total_mse")
    print(f"Best epoch: {best_epoch}, best {metric_name}: {best_val_score:.6f}")

    if config.save_prefix and config.save_final:
        torch.save(model.state_dict(), f"{config.save_prefix}_final_state.pt")
        torch.save(history, f"{config.save_prefix}_history.pt")

    return model, history

    """
    Physics run:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="physics_limited",
            mse_weight=1.0,
            darcy_weight=0.1,
            epochs=250,
            save_prefix="minimum_info/fixed_splitattn_darcy_0p1"
        )

    No physics:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="baseline_full",
            mse_weight=1.0,
            darcy_weight=0.0,
            epochs=250,
            save_prefix="minimum_info/fixed_splitattn_baseline"
        )
    """

In [ ]:
# -----------------------------------------------------------------------------
# Convenience helper for physics weight sweeps
# -----------------------------------------------------------------------------


def run_darcy_weight_sweep(
    darcy_weights,
    model_type: ModelType = "splitnet_attn",
    dataset_mode: DatasetMode = "fixed",
    epochs: int = 250,
    base_save_dir: str = "minimum_info",
    **kwargs,
):
    """
    Runs physics_limited experiments across multiple Darcy weights.

    This is for finding the best physics weight.
    It intentionally uses Limited datasets for every run in the sweep.
    """
    results = {}

    for w in darcy_weights:
        safe_w = str(w).replace(".", "p")
        prefix = f"{base_save_dir}/{dataset_mode}_physics_limited_{model_type}_darcy_{safe_w}"

        model, history = run_experiment(
            model_type=model_type,
            dataset_mode=dataset_mode,
            training_mode="physics_limited",
            mse_weight=1.0,
            darcy_weight=float(w),
            epochs=epochs,
            save_prefix=prefix,
            **kwargs,
        )

        results[w] = {
            "model": model,
            "history": history,
            "best_val_total_mse": min(history["val_total_mse"]),
            "best_val_supervised_mse": min(history["val_supervised_mse"]),
            "final_val_darcy": history["val_darcy"][-1],
        }

    return results



# -----------------------------------------------------------------------------
# Experiment plans / example runs
# -----------------------------------------------------------------------------


def run_physics_scale_tests(
    model_type: ModelType = "splitnet_attn",
    dataset_modes=("fixed", "border", "random"),
    epochs: int = 75,
    base_save_dir: str = "minimum_info/scale_tests",
):
    """
    Wide first-pass sweep to figure out the useful Darcy-weight scale.

    Your MSE may be much larger than Darcy loss, so this intentionally tests
    several orders of magnitude. These are shorter runs meant to identify a
    reasonable range before doing 250-epoch final comparisons.
    """
    darcy_weights = [
        0.0,
        0.001,
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
        1000.0,
        10000.0,
    ]

    all_results = {}

    for dataset_mode in dataset_modes:
        print(f"===== Scale test: {model_type}, {dataset_mode} =====")
        results = run_darcy_weight_sweep(
            darcy_weights,
            model_type=model_type,
            dataset_mode=dataset_mode,
            epochs=epochs,
            base_save_dir=f"{base_save_dir}/{dataset_mode}",
        )
        all_results[dataset_mode] = results

    return all_results


def run_final_comparison_grid(
    darcy_weight: float,
    epochs: int = 250,
    dataset_modes=("fixed", "border", "random"),
    model_types=("splitnet_attn", "splitnet", "unet", "attn_unet"),
    base_save_dir: str = "minimum_info/final_grid",
):
    """
    Final comparison after choosing a Darcy weight.

    For each dataset/mask option and each model type, this runs:
        1. Physics model:
           limited/masked MSE + Darcy loss over full output
        2. No-physics baseline:
           full-image MSE + no Darcy loss

    Note:
        baseline_full is unavailable for dataset_mode='random' unless you add
        a RandomDenseDatasetFull class to datasets.py. The code skips that case.
    """
    results = {}

    for dataset_mode in dataset_modes:
        for model_type in model_types:
            key_base = f"{dataset_mode}_{model_type}"
            results[key_base] = {}

            # Physics run: Limited dataset + masked MSE + Darcy loss.
            physics_prefix = (
                f"{base_save_dir}/{dataset_mode}/"
                f"{dataset_mode}_physics_limited_{model_type}_darcy_{str(darcy_weight).replace('.', 'p')}"
            )

            print(f"===== Physics run: {dataset_mode}, {model_type}, darcy={darcy_weight} =====")
            model_physics, hist_physics = run_experiment(
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="physics_limited",
                mse_weight=1.0,
                darcy_weight=darcy_weight,
                epochs=epochs,
                save_prefix=physics_prefix,
            )

            results[key_base]["physics_limited"] = {
                "model": model_physics,
                "history": hist_physics,
            }

            # Baseline run: Full dataset + full-image MSE + no Darcy.
            if dataset_mode == "random":
                print(
                    "Skipping random baseline_full because RandomDenseDatasetFull "
                    "does not exist in the current datasets.py."
                )
                continue

            baseline_prefix = (
                f"{base_save_dir}/{dataset_mode}/"
                f"{dataset_mode}_baseline_full_{model_type}_nodarcy"
            )

            print(f"===== Baseline run: {dataset_mode}, {model_type} =====")
            model_base, hist_base = run_experiment(
                model_type=model_type,
                dataset_mode=dataset_mode,
                training_mode="baseline_full",
                mse_weight=1.0,
                darcy_weight=0.0,
                epochs=epochs,
                save_prefix=baseline_prefix,
            )

            results[key_base]["baseline_full"] = {
                "model": model_base,
                "history": hist_base,
            }

    return results



def summarize_results(results):
    """
    Prints a compact summary from either run_physics_scale_tests or
    run_final_comparison_grid outputs.
    """
    print("===== Summary =====")

    for key, value in results.items():
        print(f"--- {key} ---")

        # Case 1: scale sweep result, keyed by Darcy weight.
        if all(isinstance(k, float) or isinstance(k, int) for k in value.keys()):
            for w, r in value.items():
                print(
                    f"darcy_weight={w:<10} | "
                    f"best_val_total_mse={r['best_val_total_mse']:.6f} | "
                    f"best_val_supervised_mse={r['best_val_supervised_mse']:.6f} | "
                    f"final_val_darcy={r['final_val_darcy']:.6f}"
                )

        # Case 2: final grid result, keyed by run type.
        else:
            for run_name, r in value.items():
                hist = r["history"]
                print(
                    f"{run_name:<18} | "
                    f"best_val_total_mse={min(hist['val_total_mse']):.6f} | "
                    f"best_val_supervised_mse={min(hist['val_supervised_mse']):.6f} | "
                    f"final_val_darcy={hist['val_darcy'][-1]:.6f}"
                )


if __name__ == "__main__":
    # -------------------------------------------------------------------------
    # STEP 1: Find a reasonable Darcy-weight scale.
    # -------------------------------------------------------------------------
    # Start with shorter runs across a very wide range.
    # You are looking for weights where Darcy improves/regularizes the output
    # without making MSE much worse.

    scale_results = run_physics_scale_tests(
        model_type="splitnet_attn",
        dataset_modes=("fixed", "border", "random"),
        epochs=75,
        base_save_dir="minimum_info/scale_tests",
    )
    summarize_results(scale_results)

    # -------------------------------------------------------------------------
    # STEP 2: After choosing a Darcy weight, run the final comparison grid.
    # -------------------------------------------------------------------------
    # Example: if the scale test suggests 100.0 is reasonable, use that below.

    # final_results = run_final_comparison_grid(
    #     darcy_weight=100.0,
    #     epochs=250,
    #     dataset_modes=("fixed", "border", "random"),
    #     model_types=("splitnet_attn", "splitnet", "unet", "attn_unet"),
    #     base_save_dir="minimum_info/final_grid",
    # )
    # summarize_results(final_results)



===== Scale test: splitnet_attn, fixed =====
physics_limited with darcy_weight=0.0 uses limited/masked MSE but no physics. 


100%|██████████| 75/75 [10:06<00:00,  8.08s/it]


Best epoch: 7, best val total MSE: 0.150997


100%|██████████| 75/75 [10:02<00:00,  8.03s/it]


Best epoch: 2, best val total MSE: 0.034731


...

100%|██████████| 75/75 [06:53<00:00,  5.52s/it]


Best epoch: 3, best val total MSE: 0.037008


100%|██████████| 75/75 [06:53<00:00,  5.52s/it]


Best epoch: 15, best val total MSE: 0.035174
===== Scale test: splitnet_attn, random =====
physics_limited with darcy_weight=0.0 uses limited/masked MSE but no physics. 


  8%|▊         | 6/75 [2:45:48<31:47:56, 1659.08s/it]

In [ ]:
# # Physics model: limited/masked MSE + Darcy loss over full output.
# model_fixed_darcy, hist_fixed_darcy = run_experiment(
#     model_type="splitnet_attn",
#     dataset_mode="fixed",
#     training_mode="physics_limited",
#     mse_weight=1.0,
#     darcy_weight=1.0,
#     epochs=250,
#     save_prefix="minimum_info/fixed_physics_limited_splitattn_darcy_1",
# )

# # No-physics baseline: full-image MSE + no Darcy loss.
# model_fixed_baseline, hist_fixed_baseline = run_experiment(
#     model_type="splitnet_attn",
#     dataset_mode="fixed",
#     training_mode="baseline_full",
#     mse_weight=1.0,
#     darcy_weight=0.0,
#     epochs=250,
#     save_prefix="minimum_info/fixed_baseline_full_splitattn_nodarcy",
# )

: 

: 